In [0]:
%pip install -q snowflake-connector-python pandas rapidfuzz

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.widgets.text("sf_account",   "rhxendw-yb24678")   # org-account locator
dbutils.widgets.text("sf_user",      "NEXAMART_M3")        # your Snowflake user
dbutils.widgets.text("sf_password",  "")                   # paste at run time
dbutils.widgets.text("sf_warehouse", "NEXAMART_WH")
dbutils.widgets.text("sf_role",      "NEXAMART_ENGINEER")

In [0]:
import  shutil, sqlite3
import pandas as pd
from pyspark.sql import SparkSession
 

from pyspark.sql import functions as F, Window
import sys
sys.path.append('/Workspace/Users/l1f21bsds0004@ucp.edu.pk/nexamart')
from utils_dates     import parse_date, is_parse_failure
from utils_keys      import surrogate_key, anonymous_key
from utils_anomaly   import add_anomaly_columns, flag, flag_date_parse_failure
from utils_snowflake import read_from_snowflake, write_to_snowflake
spark = SparkSession.builder.getOrCreate()
def read_bronze(t):
    return read_from_snowflake(spark, t, schema='NEXAMART_BRONZE')

def read_silver(t):
    return read_from_snowflake(spark, t, schema='NEXAMART_SILVER')

def write_silver(df, t):
    n = write_to_snowflake(df, t, schema='NEXAMART_SILVER')
    print(f'Wrote silver.{t} ({n} rows)')

def write_gold(df, t):
    n = write_to_snowflake(df, t, schema='NEXAMART_GOLD')
    print(f'Wrote gold.{t} ({n} rows)')

In [0]:
SQLITE_VOL = "/Volumes/workspace/default/nexamart/nexamart_operations.db"
SQLITE_TMP = "/tmp/nx.db"

In [0]:
shutil.copy(SQLITE_VOL, SQLITE_TMP)
print(f"✓ Copied SQLite DB to {SQLITE_TMP}")
con = sqlite3.connect(SQLITE_TMP)

✓ Copied SQLite DB to /tmp/nx.db


In [0]:
def read_sqlite(table_name):
    """Read a SQLite table into a PySpark DataFrame via pandas."""
    pdf = pd.read_sql(f"SELECT * FROM [{table_name}]", con)
    df  = spark.createDataFrame(pdf)
    print(f"  Read {df.count()} rows from SQLite [{table_name}]")
    return df

In [0]:
raw_brands          = read_sqlite("pc_brands")
raw_categories      = read_sqlite("pc_categories")
raw_condition_codes = read_sqlite("pc_condition_codes")
raw_products        = read_sqlite("pc_products")
raw_seller_listings = read_sqlite("ts_seller_listings")
 
print("\n tables loaded")

  Read 30 rows from SQLite [pc_brands]
  Read 27 rows from SQLite [pc_categories]
  Read 10 rows from SQLite [pc_condition_codes]
  Read 65 rows from SQLite [pc_products]
  Read 400 rows from SQLite [ts_seller_listings]

 tables loaded


In [0]:
print("Building silver_brands...")
 
silver_brands = (
    raw_brands
    # T3 — generate deterministic surrogate key on natural key brand_id
    .withColumn("brand_key", surrogate_key(F.col("brand_id")))
    # T1 — date standardisation: pc_brands has no date columns; N/A
    # Reorder: SK first, then business columns
    .select(
        "brand_key",
        "brand_id",
        "brand_name",
        "brand_country",
        "brand_segment",
    )
)
 
# Add 4 mandatory anomaly columns — pc_brands is clean; all rows CLEAN/CONFIRMED
silver_brands = add_anomaly_columns(silver_brands,
                                    default_status="CLEAN",
                                    default_certainty="CONFIRMED")
 
print(f"  Row count: {silver_brands.count()}  (expected 30)")
silver_brands.show(5, truncate=False)
 
# Validate surrogate key uniqueness
dup_brands = silver_brands.groupBy("brand_key").count().filter(F.col("count") > 1)
assert dup_brands.count() == 0, "DUPLICATE brand_key detected!"
print("  ✓ brand_key is unique (no duplicates)")

Building silver_brands...
  Row count: 30  (expected 30)
+----------------------------------------------------------------+--------+----------+-------------+-------------+------------+-------------------+-------------------+----------------------+
|brand_key                                                       |brand_id|brand_name|brand_country|brand_segment|anomaly_flag|anomaly_reason_code|data_quality_status|metric_certainty_level|
+----------------------------------------------------------------+--------+----------+-------------+-------------+------------+-------------------+-------------------+----------------------+
|6b86b273ff34fce19d6b804eff5a3f5747ada4eaa22f1d49c01e52ddb7875b4b|1       |Apple     |USA          |PREMIUM      |false       |NULL               |CLEAN              |CONFIRMED             |
|d4735e3a265e16eee03f59718b9b5d03019c07d8b6c51f90da3a666eec13ab35|2       |Samsung   |South Korea  |MID_PREMIUM  |false       |NULL               |CLEAN              |CONFIRMED   

In [0]:
# Use inline version to avoid external module dbutils issue
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
import pandas as pd

def _sf_conn_inline():
    return snowflake.connector.connect(
        account   = dbutils.widgets.get("sf_account"),
        user      = dbutils.widgets.get("sf_user"),
        password  = dbutils.widgets.get("sf_password"),
        warehouse = dbutils.widgets.get("sf_warehouse"),
        role      = dbutils.widgets.get("sf_role"),
        database  = "NEXAMART_DW",
    )

def write_to_snowflake_inline(df, table_name, schema="NEXAMART_SILVER"):
    """Inline version that works with Spark Connect."""
    # Workaround for Spark Connect: use collect() instead of toPandas()
    data = df.collect()
    columns = df.columns
    pdf = pd.DataFrame([row.asDict() for row in data], columns=columns)
    pdf.columns = [c.upper() for c in pdf.columns]
    with _sf_conn_inline() as ctx:
        cursor = ctx.cursor()
        cursor.execute(f"USE SCHEMA {schema}")
        success, nchunks, nrows, _ = write_pandas(
            ctx, pdf,
            table_name       = table_name.upper(),
            schema           = schema,
            database         = "NEXAMART_DW",
            auto_create_table= True,
            overwrite        = True,
            quote_identifiers= False,
        )
    print(f"  ✓ Wrote {nrows} rows → NEXAMART_DW.{schema}.{table_name.upper()}")
    return nrows

write_to_snowflake_inline(silver_brands, "silver_brands", schema="NEXAMART_SILVER")

  ✓ Wrote 30 rows → NEXAMART_DW.NEXAMART_SILVER.SILVER_BRANDS


30

In [0]:
print("Building silver_categories...")
 
silver_categories = (
    raw_categories
    .withColumn("category_key", surrogate_key(F.col("category_id")))
    # Also generate parent_category_key (nullable — NULL for root categories)
    .withColumn("parent_category_key",
                F.when(F.col("parent_category_id").isNotNull(),
                       surrogate_key(F.col("parent_category_id")))
                 .otherwise(F.lit(None).cast("string")))
    .select(
        "category_key",
        "category_id",
        "category_name",
        "parent_category_id",
        "parent_category_key",
        "level",
    )
)
 
silver_categories = add_anomaly_columns(silver_categories)
 
print(f"  Row count: {silver_categories.count()}  (expected 27)")
silver_categories.show(5, truncate=False)
 
dup_cats = silver_categories.groupBy("category_key").count().filter(F.col("count") > 1)
assert dup_cats.count() == 0, "DUPLICATE category_key detected!"
print("  ✓ category_key is unique")
 
write_to_snowflake_inline(silver_categories, "silver_categories", schema="NEXAMART_SILVER")

Building silver_categories...
  Row count: 27  (expected 27)
+----------------------------------------------------------------+-----------+------------------+------------------+----------------------------------------------------------------+-----+------------+-------------------+-------------------+----------------------+
|category_key                                                    |category_id|category_name     |parent_category_id|parent_category_key                                             |level|anomaly_flag|anomaly_reason_code|data_quality_status|metric_certainty_level|
+----------------------------------------------------------------+-----------+------------------+------------------+----------------------------------------------------------------+-----+------------+-------------------+-------------------+----------------------+
|6b86b273ff34fce19d6b804eff5a3f5747ada4eaa22f1d49c01e52ddb7875b4b|1          |Electronics       |NULL              |NULL                           

27

In [0]:
print("Building silver_condition_codes...")
 
silver_condition_codes = (
    raw_condition_codes
    .withColumn("condition_code_key", surrogate_key(F.col("code")))
    .select(
        "condition_code_key",
        "code",
        "description",
    )
)
 
silver_condition_codes = add_anomaly_columns(silver_condition_codes)
 
print(f"  Row count: {silver_condition_codes.count()}  (expected 10)")
silver_condition_codes.show(truncate=False)
 
dup_cond = silver_condition_codes.groupBy("condition_code_key").count().filter(F.col("count")>1)
assert dup_cond.count() == 0, "DUPLICATE condition_code_key detected!"
print("  ✓ condition_code_key is unique")
 
write_to_snowflake_inline(silver_condition_codes, "silver_condition_codes", schema="NEXAMART_SILVER")

Building silver_condition_codes...
  Row count: 10  (expected 10)
+----------------------------------------------------------------+--------+------------------------------+------------+-------------------+-------------------+----------------------+
|condition_code_key                                              |code    |description                   |anomaly_flag|anomaly_reason_code|data_quality_status|metric_certainty_level|
+----------------------------------------------------------------+--------+------------------------------+------------+-------------------+-------------------+----------------------+
|a253ff09c5a8678e1fd1962b2c329245e139e45f9cc6ced4e5d7ad42c4108fc0|NEW     |Brand New — Factory Sealed    |false       |NULL               |CLEAN              |CONFIRMED             |
|11aa8a2c2243c89a0422ec2589cb2c4507980350ce74094ee61df683b1581cb3|NEW_OPEN|New — Box Opened, Unused      |false       |NULL               |CLEAN              |CONFIRMED             |
|9e94518d6ef9af2ce4

10

In [0]:
print("Building silver_products...")
 
# T1 — date standardisation
# pc_products dates are already ISO YYYY-MM-DD strings → cast to DateType
products_dated = (
    raw_products
    .withColumn("created_date_parsed",
                F.to_date(F.col("created_date"), "yyyy-MM-dd"))
    .withColumn("last_updated_parsed",
                F.to_date(F.col("last_updated"), "yyyy-MM-dd"))
)
 
# T3 — surrogate key on sku
products_keyed = products_dated.withColumn("product_key", surrogate_key(F.col("sku")))
 
# Join to silver_brands to bring in brand_key (FK resolution)
products_with_brand = (
    products_keyed
    .join(
        silver_brands.select("brand_id", "brand_key").withColumnRenamed("brand_key","brand_key_fk"),
        on="brand_id",
        how="left"
    )
)
 
# Join to silver_categories to bring in category_key (FK resolution)
products_with_cat = (
    products_with_brand
    .join(
        silver_categories.select("category_id","category_key").withColumnRenamed("category_key","category_key_fk"),
        on="category_id",
        how="left"
    )
)
 
# Join to silver_condition_codes to bring in condition_code_key
products_with_cond = (
    products_with_cat
    .join(
        silver_condition_codes.select("code","condition_code_key").withColumnRenamed("code","condition_code_raw"),
        F.col("condition_code") == F.col("condition_code_raw"),
        how="left"
    )
)
 
silver_products = (
    products_with_cond
    .select(
        "product_key",
        "product_id",
        "sku",
        "product_name",
        "category_id",
        F.col("category_key_fk").alias("category_key"),
        "brand_id",
        F.col("brand_key_fk").alias("brand_key"),
        "condition_code",
        "condition_code_key",
        "standard_cost",
        "rrp",
        "weight_kg",
        "is_campaign_product",
        "is_active",
        F.col("created_date_parsed").alias("created_date"),
        F.col("last_updated_parsed").alias("last_updated"),
        "supplier_code",
    )
)
 
silver_products = add_anomaly_columns(silver_products)
 
# Flag date parse failures (if any)
silver_products = flag(
    silver_products,
    condition=F.col("created_date").isNull() & F.col("product_id").isNotNull(),
    reason_code="DATE_PARSE_FAIL",
    status="FLAGGED_ANOMALY",
    certainty="UNRELIABLE"
)
 
print(f"  Row count: {silver_products.count()}  (expected 65)")
silver_products.select("product_key","sku","product_name","category_key","brand_key").show(5, truncate=False)
 
dup_prod = silver_products.groupBy("product_key").count().filter(F.col("count") > 1)
assert dup_prod.count() == 0, "DUPLICATE product_key detected!"
print("  ✓ product_key is unique")
 
write_to_snowflake_inline(silver_products, "silver_products", schema="NEXAMART_SILVER")

Building silver_products...
  Row count: 65  (expected 65)
+----------------------------------------------------------------+------------+----------------------------------------+----------------------------------------------------------------+----------------------------------------------------------------+
|product_key                                                     |sku         |product_name                            |category_key                                                    |brand_key                                                       |
+----------------------------------------------------------------+------------+----------------------------------------+----------------------------------------------------------------+----------------------------------------------------------------+
|63b21aa00048e0fa8daafd8663ed1e3bb27d028976cc82551cd2b90a50d3e706|NX-TECH-0001|NexaBook Pro 15 — Core i7 16GB 512GB SSD|d4735e3a265e16eee03f59718b9b5d03019c07d8b6c51f90da3a666eec13ab35|ef2

65

In [0]:
print("Building silver_price_history (schema-only — 0 rows)...")
 
from pyspark.sql.types import (StructType, StructField,
                               LongType, StringType, DateType, DoubleType,
                               BooleanType)
 
ph_schema = StructType([
    StructField("price_history_key",      StringType(),  False),  # SK
    StructField("history_id",             LongType(),    True),
    StructField("sku",                    StringType(),  True),
    StructField("product_key",            StringType(),  True),   # FK → silver_products
    StructField("effective_date",         DateType(),    True),
    StructField("rrp",                    DoubleType(),  True),
    StructField("standard_cost",          DoubleType(),  True),
    StructField("change_reason",          StringType(),  True),
    # 4 mandatory anomaly columns
    StructField("anomaly_flag",           BooleanType(), False),
    StructField("anomaly_reason_code",    StringType(),  True),
    StructField("data_quality_status",    StringType(),  False),
    StructField("metric_certainty_level", StringType(),  False),
])
 
silver_price_history = spark.createDataFrame([], ph_schema)
print(f"  Row count: {silver_price_history.count()}  (expected 0 — schema only)")
 
write_to_snowflake_inline(silver_price_history, "silver_price_history", schema="NEXAMART_SILVER")
print("  ✓ silver_price_history written (0 rows, correct schema)")
print("  NOTE: pc_price_history is a forward-looking table; rows expected in Milestone 2")

Building silver_price_history (schema-only — 0 rows)...
  Row count: 0  (expected 0 — schema only)
  ✓ Wrote 0 rows → NEXAMART_DW.NEXAMART_SILVER.SILVER_PRICE_HISTORY
  ✓ silver_price_history written (0 rows, correct schema)
  NOTE: pc_price_history is a forward-looking table; rows expected in Milestone 2


In [0]:
print("Building silver_product_master (T5 — exact SKU pass)...")
 
# Step 1: Prepare seller listings for join
seller_for_t5 = (
    raw_seller_listings
    .select(
        "listing_id",
        "seller_id",
        F.col("nexamart_sku_ref").alias("seller_sku_ref"),
        F.col("seller_product_title").alias("seller_product_name"),
        F.col("seller_sku").alias("seller_internal_sku"),
        "asking_price",
        "condition_code",
    )
)
 
# Step 2: INNER JOIN — catalogue SKUs that have at least one seller listing
exact_matched = (
    silver_products
    .join(seller_for_t5, silver_products.sku == seller_for_t5.seller_sku_ref, how="inner")
    .select(
        # Canonical product key: deterministic on catalogue SKU
        surrogate_key(silver_products.sku).alias("canonical_product_key"),
        silver_products.sku.alias("sku"),
        # Catalogue name ALWAYS wins
        silver_products.product_name.alias("product_name"),
        silver_products.brand_key.alias("brand_key"),
        silver_products.category_key.alias("category_key"),
        silver_products.condition_code_key.alias("condition_code_key"),
        silver_products.standard_cost.alias("cogs"),
        silver_products.rrp.alias("list_price"),
        # Seller reference fields (for audit)
        F.col("listing_id").alias("match_source_listing_id"),
        F.col("seller_id").alias("match_source_seller_id"),
        F.col("seller_product_name").alias("seller_product_name_raw"),
        # Match metadata
        F.lit(1.00).alias("match_confidence"),
        F.lit("EXACT_SKU").alias("match_pass"),
    )
)
 
# Step 3: LEFT ANTI JOIN — seller listings whose SKU is NOT in catalogue
# These are unknown/rogue SKU references → match_confidence = 0.0
seller_only_skus = (
    seller_for_t5
    .join(silver_products, seller_for_t5.seller_sku_ref == silver_products.sku, how="left_anti")
    .select(
        surrogate_key(F.col("seller_sku_ref")).alias("canonical_product_key"),
        F.col("seller_sku_ref").alias("sku"),
        F.col("seller_product_name").alias("product_name"),
        F.lit(None).cast("string").alias("brand_key"),
        F.lit(None).cast("string").alias("category_key"),
        F.lit(None).cast("string").alias("condition_code_key"),
        F.lit(None).cast("double").alias("cogs"),
        F.col("asking_price").alias("list_price"),
        F.col("listing_id").alias("match_source_listing_id"),
        F.col("seller_id").alias("match_source_seller_id"),
        F.col("seller_product_name").alias("seller_product_name_raw"),
        F.lit(0.0).alias("match_confidence"),
        F.lit("SKU_NOT_IN_CATALOGUE").alias("match_pass"),
    )
)
 
# Step 4: UNION all rows into silver_product_master
silver_product_master = exact_matched.unionByName(seller_only_skus)
 
# Add 4 mandatory anomaly columns — default CLEAN/CONFIRMED
silver_product_master = add_anomaly_columns(silver_product_master)
 
print(f"  Exact-matched rows : {exact_matched.count()}")
print(f"  Seller-only rows   : {seller_only_skus.count()}")
print(f"  Total rows         : {silver_product_master.count()}")

Building silver_product_master (T5 — exact SKU pass)...
  Exact-matched rows : 400
  Seller-only rows   : 0
  Total rows         : 400


In [0]:
print("Flagging A9 — SKU_PRODUCT_MISMATCH...")
 
def word_overlap_ratio(cat_col, sel_col):
    """
    UDF-free Spark implementation of word overlap ratio.
    Lower-cases, strips punctuation, splits on whitespace,
    then computes |intersection(cat_words, sel_words)| / |cat_words|.
    Uses Spark array functions — no Python UDF needed.
    """
    # Normalise: lower, replace em-dash and hyphen with space
    cat_clean = F.regexp_replace(F.lower(cat_col), r"[—\-]", " ")
    sel_clean = F.regexp_replace(F.lower(sel_col), r"[—\-]", " ")
    # Split on whitespace → arrays
    cat_words = F.split(F.trim(cat_clean), r"\s+")
    sel_words = F.split(F.trim(sel_clean), r"\s+")
    # Intersection size
    intersect_size = F.size(F.array_intersect(cat_words, sel_words))
    cat_size       = F.size(cat_words)
    # Ratio: safe division
    return F.when(cat_size > 0, intersect_size / cat_size).otherwise(F.lit(0.0))
 
# Only flag EXACT_SKU rows (seller-only rows have no catalogue name to compare)
overlap_col = F.when(
    F.col("match_pass") == "EXACT_SKU",
    word_overlap_ratio(F.col("product_name"), F.col("seller_product_name_raw"))
).otherwise(F.lit(1.0))   # seller-only rows: don't flag as mismatch (different anomaly)
 
silver_product_master = silver_product_master.withColumn("_name_overlap_ratio", overlap_col)
 
# Flag rows where seller title is wildly different (overlap < 35%)
mismatch_condition = (
    (F.col("match_pass") == "EXACT_SKU") &
    (F.col("_name_overlap_ratio") < 0.35)
)
 
silver_product_master = flag(
    silver_product_master,
    condition    = mismatch_condition,
    reason_code  = "SKU_PRODUCT_MISMATCH",
    status       = "FLAGGED_ANOMALY",
    certainty    = "UNRELIABLE",
)
 
# Flag seller-only rows (SKU_NOT_IN_CATALOGUE)
not_in_cat_condition = (F.col("match_pass") == "SKU_NOT_IN_CATALOGUE")
silver_product_master = flag(
    silver_product_master,
    condition   = not_in_cat_condition,
    reason_code = "SKU_NOT_IN_CATALOGUE",
    status      = "FLAGGED_ANOMALY",
    certainty   = "UNRELIABLE",
)
 
# Drop the internal helper column
silver_product_master = silver_product_master.drop("_name_overlap_ratio")
 
# Final count
print(f"\n  ── silver_product_master summary ──")
silver_product_master.groupBy("match_pass","anomaly_flag","anomaly_reason_code").count().show(truncate=False)
 
# Show the actual A9 flagged rows
print("\n  A9 flagged rows (SKU_PRODUCT_MISMATCH):")
(silver_product_master
 .filter(F.col("anomaly_reason_code").contains("SKU_PRODUCT_MISMATCH"))
 .select("sku","product_name","seller_product_name_raw","match_confidence","anomaly_reason_code")
 .show(truncate=False))
 
# Validation: every catalogue product (65) must have at least one row
cat_skus_in_master = silver_product_master.filter(F.col("match_confidence")==1.00).select("sku").distinct().count()
print(f"\n  Catalogue SKUs represented: {cat_skus_in_master} (expected 65)")
assert cat_skus_in_master == 65, "Not all 65 catalogue SKUs are in silver_product_master!"
 
write_to_snowflake_inline(silver_product_master, "silver_product_master", schema="NEXAMART_SILVER")
print("\n✓ silver_product_master committed — ping Lead that M3 Silver is done!")

Flagging A9 — SKU_PRODUCT_MISMATCH...

  ── silver_product_master summary ──
+----------+------------+--------------------+-----+
|match_pass|anomaly_flag|anomaly_reason_code |count|
+----------+------------+--------------------+-----+
|EXACT_SKU |true        |SKU_PRODUCT_MISMATCH|1    |
|EXACT_SKU |false       |NULL                |399  |
+----------+------------+--------------------+-----+


  A9 flagged rows (SKU_PRODUCT_MISMATCH):
+------------+----------------------------------------+---------------------------------------------------+----------------+--------------------+
|sku         |product_name                            |seller_product_name_raw                            |match_confidence|anomaly_reason_code |
+------------+----------------------------------------+---------------------------------------------------+----------------+--------------------+
|NX-TECH-0001|NexaBook Pro 15 — Core i7 16GB 512GB SSD|Premium Protective Phone Case Cover — Universal Fit|1.0             